# Notebook to clean up the Medicare Part D file and generate the data for RQ2

In [15]:
from pyspark import SparkContext
import pyspark
import csv

sc = pyspark.SparkContext().getOrCreate()

In [16]:
raw_data = sc.textFile('../data/MedicarePartDPrescribersByProvider/2024/mup_dpr_ry26_p04_v20_dy24_npi.csv').mapPartitions(lambda lines: csv.reader(lines))


In [17]:
all_header = raw_data.first()
all_header[:10]

['Prscrbr_NPI',
 'Prscrbr_Last_Org_Name',
 'Prscrbr_First_Name',
 'Prscrbr_MI',
 'Prscrbr_Crdntls',
 'Prscrbr_Ent_Cd',
 'Prscrbr_St1',
 'Prscrbr_St2',
 'Prscrbr_City',
 'Prscrbr_State_Abrvtn']

In [18]:
print(f"Shape: {raw_data.count()}x{len(raw_data.first())}")
counts = raw_data.map(lambda x: len(x)).countByValue()
print(counts)
sum(counts.values())

Shape: 1416884x84


defaultdict(<class 'int'>, {84: 1416884})


1416884

In [19]:
raw_data.take(5)

[['Prscrbr_NPI',
  'Prscrbr_Last_Org_Name',
  'Prscrbr_First_Name',
  'Prscrbr_MI',
  'Prscrbr_Crdntls',
  'Prscrbr_Ent_Cd',
  'Prscrbr_St1',
  'Prscrbr_St2',
  'Prscrbr_City',
  'Prscrbr_State_Abrvtn',
  'Prscrbr_State_FIPS',
  'Prscrbr_Zip5',
  'Prscrbr_RUCA',
  'Prscrbr_RUCA_Desc',
  'Prscrbr_Cntry',
  'Prscrbr_Type',
  'Prscrbr_Type_Src',
  'Tot_Clms',
  'Tot_30day_Fills',
  'Tot_Drug_Cst',
  'Tot_Day_Suply',
  'Tot_Benes',
  'GE65_Sprsn_Flag',
  'GE65_Tot_Clms',
  'GE65_Tot_30day_Fills',
  'GE65_Tot_Drug_Cst',
  'GE65_Tot_Day_Suply',
  'GE65_Bene_Sprsn_Flag',
  'GE65_Tot_Benes',
  'Brnd_Sprsn_Flag',
  'Brnd_Tot_Clms',
  'Brnd_Tot_Drug_Cst',
  'Gnrc_Sprsn_Flag',
  'Gnrc_Tot_Clms',
  'Gnrc_Tot_Drug_Cst',
  'Othr_Sprsn_Flag',
  'Othr_Tot_Clms',
  'Othr_Tot_Drug_Cst',
  'MAPD_Sprsn_Flag',
  'MAPD_Tot_Clms',
  'MAPD_Tot_Drug_Cst',
  'PDP_Sprsn_Flag',
  'PDP_Tot_Clms',
  'PDP_Tot_Drug_Cst',
  'LIS_Sprsn_Flag',
  'LIS_Tot_Clms',
  'LIS_Drug_Cst',
  'NonLIS_Sprsn_Flag',
  'NonLIS_Tot_Clms

## Columns to be extracted
| Term name | Column name | Definition |
| :--- | :--- | :--- |
| National Provider Identifier | Prscrbr_NPI | National Provider Identifier (NPI) for the performing provider on the claim.|
|State Code of the Provider | Prscrbr_State_Abrvtn | The state where the provider is located, as reported in NPPES. The fifty U.S. states and the District of Columbia are reported by the state postal abbreviation. The following values are used for other areas:|
|Zip Code of the Provider (first five digits) | Prscrbr_Zip5 | The first 5 digits of the provider’s ZIP code, as reported in NPPES.|
| Provider Specialty Type | Prscrbr_Type | Derived from the Medicare provider/supplier specialty code reported on the NPI’s Part B claims. For providers that have more than one Medicare specialty code reported on their claims, the Medicare specialty code associated with the largest number of services is reported. Where a prescriber’s NPI did not have associated Part B claims, the taxonomy code associated with the NPI in NPPES is mapped to a Medicare specialty code using an external crosswalk published here: https://data.cms.gov/provider-characteristics/medicare-provider-supplier-enrollment/medicare-provider-and-supplier-taxonomy-crosswalk. <br> For any taxonomy codes that could not be mapped to a Medicare specialty code, the taxonomy classification description from the National Uniform Claim Committee (NUCC) taxonomy code set is used. For more information on the NUCC taxonomy code set, please visit: http://www.nucc.org |
|Entity Type of the Provider Prscrbr_Ent_Cd | Prscrbr_Ent_Cd | Type of entity reported in NPPES. An entity code of ‘I’ identifies referring providers registered as individuals and an Prscrbr_Ent_Cof ‘O’ identifies referring providers registered as organizations.|
| Number of Medicare Part D Claims, Including Refills | Tot_Clms | The number of Medicare Part D claims. This includes original prescriptions and refills. Aggregated records based on total_claim_count fewer than 11 are not included in the data file.|
|Number of Standardized 30-Day Fills, Including Refills | Tot_30day_Fills | The aggregate number of Medicare Part D standardized 30-day fills. The standardized 30-day fill is derived from the number of days supplied on each Part D claim divided by 30. Standardized 30day fill values less than 1.0 were bottom-coded with a value of 1.0 and standardized 30-day fill values greater than 12.0 were top-coded with a value of 12.0.|
|Aggregate Cost Paid for All Claims | Tot_Drug_Cst | The aggregate drug cost paid for all associated claims. This amounincludes ingredient cost, dispensing fee, sales tax, and any applicable vaccine administration fees and is based on the amounts paid by the Part D plan, Medicare beneficiary, government subsidies, and any other third-party payers.|
| Aggregate Cost Paid for Brand-Name Drugs | Brnd_Tot_Drug_Cst | Aggregate drug cost paid for brand-name drugs. This amount includes ingredient cost, dispensing fee, sales tax, and any applicable vaccine administration fees and is based on the amounts paid by the Part D plan, Medicare beneficiary, government subsidies, and any other third-party payers. A drug is classified as “brand” using the FDA approval category of NDA, NDA authorized generic, or BLA. If Brnd_Tot_Clms is suppressed this variable is suppressed. A blank indicates the value is suppressed. See Brnd_Sprsn_Flag regarding suppression of data.|
|Aggregate Cost Paid for Generic Drugs | Gnrc_Tot_Drug_Cst | Aggregate cost paid for generic drugs. This amount includes ingredient cost, dispensing fee, sales tax, and any applicable vaccine administration fees and is based on the amounts paid by the Part D plan, Medicare beneficiary, government subsidies, and any other third-party payers. A drug is classified as “generic” using the FDA approval category of ANDA. If Gnrc_Tot_Clms is suppressed this variable is suppressed. A blank indicates the value is suppressed. See Gnrc_Sprsn_Flag regarding suppression of data|

In [20]:
needed_columns = [
    "Prscrbr_NPI", 
    "Prscrbr_Ent_Cd",
    "Prscrbr_State_Abrvtn",
    "Prscrbr_Zip5",
    "Prscrbr_Type",
    "Prscrbr_Type_Src",
    "Tot_Clms",
    "Tot_30day_Fills",
    "Tot_Drug_Cst",
    "Brnd_Tot_Drug_Cst",
    "Gnrc_Tot_Drug_Cst",
    "Tot_Benes"
    ]


column_indexes = [all_header.index(x) for x in needed_columns]

In [21]:
raw_data_wo_header = raw_data.filter(lambda x: x != all_header)


In [22]:

def strip_columns(row: list[str], indexes) -> list[str]:
        res = []
        for i, val in enumerate(row):
            if i not in indexes:
                continue
            res.append(val)
        return res
data = raw_data_wo_header.map(lambda x: strip_columns(x, column_indexes))
header = [all_header[i] for i in column_indexes]
print(header)
data.first()

['Prscrbr_NPI', 'Prscrbr_Ent_Cd', 'Prscrbr_State_Abrvtn', 'Prscrbr_Zip5', 'Prscrbr_Type', 'Prscrbr_Type_Src', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst', 'Brnd_Tot_Drug_Cst', 'Gnrc_Tot_Drug_Cst', 'Tot_Benes']


['1053753350',
 'I',
 'CA',
 '92832',
 'Dentist',
 'NPPES-Specialty',
 '43.0',
 '43.066667',
 '255.88000000000002',
 '21.0',
 '0.0',
 '255.88000000000002']

In [23]:

column_name_to_index_map = {i: header[i] for i in range(len(header))}
columns_types = [
    int, str, str, str, str, str, int, float, float, int, float, float
]
column_name_to_index_map

{0: 'Prscrbr_NPI',
 1: 'Prscrbr_Ent_Cd',
 2: 'Prscrbr_State_Abrvtn',
 3: 'Prscrbr_Zip5',
 4: 'Prscrbr_Type',
 5: 'Prscrbr_Type_Src',
 6: 'Tot_Clms',
 7: 'Tot_30day_Fills',
 8: 'Tot_Drug_Cst',
 9: 'Brnd_Tot_Drug_Cst',
 10: 'Gnrc_Tot_Drug_Cst',
 11: 'Tot_Benes'}

In [24]:
def convert_types(row: list[str], types: list[type]):
        res = []
        for i, val in enumerate(row):
            try:
                if val == "":
                     res.append(None)
                elif column_name_to_index_map.get(i, "") == "Prscrbr_Zip5":
                    zip = str(int(val)).zfill(5)
                    res.append(zip)
                elif types[i] is int:
                    res.append(int(float(val)))
                else:
                    target_type = types[i]            
                    res.append(target_type(val))
            except BaseException:
                 raise ValueError(f"Failed to cast {header[i]} {val} to {types[i]}")
                  
        return res

typed_data = data.map(lambda x: convert_types(x, columns_types))
typed_data.take(3)

[[1053753350,
  'I',
  'CA',
  '92832',
  'Dentist',
  'NPPES-Specialty',
  43,
  43.066667,
  255.88000000000002,
  21,
  0.0,
  255.88000000000002],
 [1053753434,
  'I',
  'KY',
  '41017',
  'Physician Assistant',
  'Claim-Specialty',
  118,
  118.0,
  464.01999999999964,
  99,
  0.0,
  464.01999999999964],
 [1053753442,
  'I',
  'FL',
  '32117',
  'Internal Medicine',
  'Claim-Specialty',
  2055,
  3449.4999970000003,
  419734.92,
  519,
  385479.56,
  34255.36000000002]]

In [ ]:
import re
state_index = header.index("Prscrbr_State_Abrvtn")
# typed_data.map(lambda x: x[state_index]).countByValue()
zip_index = header.index("Prscrbr_Zip5")
ca_zip_pattern = r'^9(?:[0-5]\d|6[01])\d{2}'
california_data = typed_data.filter(lambda x: x[state_index] == "CA")
california_data = california_data.filter(lambda x: x is not None and re.fullmatch(ca_zip_pattern, x[zip_index]))
california_data.count()

143204

In [26]:
sc.stop()